In [6]:
import sys
print("Python:", sys.version)
print("Location:", sys.executable)
# You should see .venv in the path

Python: 3.13.2 (tags/v3.13.2:4f8bb39, Feb  4 2025, 15:23:48) [MSC v.1942 64 bit (AMD64)]
Location: d:\final_chatbot_for_summarize_new\.venv\Scripts\python.exe


In [7]:
import os
from anthropic import Anthropic
from dotenv import load_dotenv

In [8]:
# BOILERPLATE — same pattern every project, never changes
load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

if not ANTHROPIC_API_KEY:
    print("❌ Key not found — check your .env file!")
else:
    print("✅ API key loaded!")
    print("   Starts with:", ANTHROPIC_API_KEY[:10] + "****")

✅ API key loaded!
   Starts with: sk-ant-api****


In [9]:
# BOILERPLATE — your connection to Claude
# Same idea as requests in Module 1, just different library
client = Anthropic()

print("✅ Claude client ready!")

✅ Claude client ready!


In [10]:
# YOU WRITE THIS → system prompt (Claude's personality)
system_prompt = "You are "

In [11]:
def summarize_article(article_text : str , ticker : str = "" ) -> dict :
    """
    Send a news article to Claude and get back a summary
    in both Thai and English.
    
    Args:
        article_text: the raw news article text
        ticker: optional stock ticker to give context e.g. "AAPL"
    
    Returns:
        dict with 'english' and 'thai' summaries
    """
    system_prompt = """ You are a financial analyst assistant specializing in investment
    news. You summarize news article clearly and concisely.Always respond in this exact format:

    English Summary : 
    [2-3 sentence summary in English]

    Thai Summary :
    [2-3 sentence summary in Thai]

    Impact:
    [One sentence on how this effects investors]"""

    user_prompt = f"""Please summarize this financial news article.
    {"Focus on impact to" + ticker + " investors." if ticker else ""}

    Article:
    {article_text}"""
    
    response = client.messages.create(
        model = 'claude-haiku-4-5-20251001',
        max_tokens = 1024 , 
        system = system_prompt,
        messages =[{"role" : "user" , "content" : user_prompt}]
    )
    raw_text = response.content[0].text

    result = {
    "raw": raw_text,
    "ticker": ticker,
    "tokens_used": response.usage.input_tokens + response.usage.output_tokens
    }
    return result    


In [12]:
system_prompt = "You are a helpful assistant. Keep you answers under 2 sentences"

user_prompt = "What is Nvidia and why do investors care about it"

response = client.messages.create(
    model = "claude-haiku-4-5-20251001" ,
    max_tokens = 1024 ,
    system = system_prompt ,
    messages = [
        {'role' : 'user' ,'content' : user_prompt}
        ]
)

result = response.content[0].text
print('Claude says:')
print(result)

Claude says:
Nvidia is a leading semiconductor company that designs powerful GPUs (graphics processors) used in data centers, AI, gaming, and autonomous vehicles. Investors care about it because it dominates the AI chip market, has shown explosive growth and profitability, and is seen as a key beneficiary of the AI revolution transforming technology and industry.


In [13]:
print('Model used   :', response.model)
print('Stop reason   :', response.stop_reason)
print('Token sent', response.usage.input_tokens)
print('Token used', response.usage.output_tokens)
print('Total tokens' ,response.usage.input_tokens + response.usage.output_tokens)
print(response.content[0].text)



Model used   : claude-haiku-4-5-20251001
Stop reason   : end_turn
Token sent 32
Token used 73
Total tokens 105
Nvidia is a leading semiconductor company that designs powerful GPUs (graphics processors) used in data centers, AI, gaming, and autonomous vehicles. Investors care about it because it dominates the AI chip market, has shown explosive growth and profitability, and is seen as a key beneficiary of the AI revolution transforming technology and industry.


In [14]:
def summarize_article(article_text: str, ticker: str = "") -> dict:
    """
    Send a news article to Claude, get back a bilingual summary.
    
    Args:
        article_text : the raw news article text
        ticker       : stock symbol e.g. "NVDA" (optional)
    
    Returns:
        dict with summary text and token usage
    """

    # YOU WRITE THIS → Claude's personality & output rules
    system_prompt = """You are a financial analyst assistant.
You summarize news articles clearly and concisely.
Always respond in this exact format:

ENGLISH SUMMARY:
[2-3 sentence summary in English]

THAI SUMMARY:
[2-3 sentence summary in Thai]

IMPACT:
[One sentence on how this affects investors]"""

    # YOU WRITE THIS → the task you give Claude
    user_prompt = f"""Please summarize this financial news article.
{"Focus on impact to " + ticker + " investors." if ticker else ""}

Article:
{article_text}"""

    # BOILERPLATE → API call, same structure as Cell 5
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        system=system_prompt,
        messages=[
            {"role": "user", "content": user_prompt}
        ]
    )

    # BOILERPLATE → extract and return results
    return {
        "summary"        : response.content[0].text,
        "ticker"         : ticker,
        "input_tokens"   : response.usage.input_tokens,
        "output_tokens"  : response.usage.output_tokens
    }

print("✅ summarize_article() function ready!")

✅ summarize_article() function ready!


In [15]:
fake_article = """
NVIDIA reported record quarterly earnings today, with revenue 
surging 122% year-over-year to $22.1 billion. The results were 
driven by explosive demand for its H100 AI chips from major 
cloud providers including Microsoft, Google, and Amazon. 
CEO Jensen Huang said demand continues to far exceed supply, 
and the company expects strong growth to continue into 2025.
"""

result = summarize_article(fake_article , ticker = 'NVDA')

print('Summary')
print(result['summary'])
print('')
print("=== STATS ===")
print('Ticker  : ',result['ticker'])
print('Input Tokens:',result['input_tokens'])
print('Output Tokens:',result['output_tokens'])

Summary
ENGLISH SUMMARY:
NVIDIA achieved record quarterly revenue of $22.1 billion, representing a 122% year-over-year surge, primarily driven by massive demand for its H100 AI chips from major cloud providers. CEO Jensen Huang indicated that demand continues to substantially exceed supply, with management projecting sustained strong growth throughout 2025.

THAI SUMMARY:
NVIDIA บรรลุรายได้รายไตรมาสสูงสุดที่ 22.1 พันล้านดอลลาร์ เพิ่มขึ้น 122% เมื่อเทียบกับปีที่แล้ว โดยขับเคลื่อนด้วยความต้องการสูงมากสำหรับชิป H100 AI ของบริษัท ซีอีโอ Jensen Huang ระบุว่าความต้องการยังคงเกินอุปทานอย่างมาก และคาดว่าการเติบโตที่แข็งแกร่งจะยังคงดำเนินต่อไปในปี 2025

IMPACT:
NVDA investors benefit from accelerating AI chip demand, supply constraints supporting premium pricing, and management's positive 2025 outlook, which should provide strong support for stock valuation and earnings growth.

=== STATS ===
Ticker  :  NVDA
Input Tokens: 193
Output Tokens: 343


In [17]:
general_article = """
The Federal Reserve announced today it will hold interest rates 
steady at 5.25%, citing ongoing concerns about inflation. 
Fed Chair Jerome Powell hinted that rate cuts may begin 
in late 2025 if inflation continues to cool.
"""
result = summarize_article(general_article)
print(result['summary'])

ENGLISH SUMMARY:
The Federal Reserve maintained interest rates at 5.25% and expressed continued concern about inflation levels. Fed Chair Jerome Powell suggested that interest rate cuts could potentially begin in late 2025 if inflationary pressures continue to decline.

THAI SUMMARY:
ธนาคารกลางสหรัฐฯ ประกาศให้อัตราดอกเบี้ยคงที่ที่ 5.25% และแสดงความกังวลต่อเนื่องเกี่ยวกับอัตราเงินเฟ้อ ประธาน Fed เจอโรม พาวเวลล์ชี้ให้เห็นว่าอาจมีการลดอัตราดอกเบี้ยในปลายปี 2568 หากแรงกดดันจากเงินเฟ้อลดลงต่อไป

IMPACT:
Investors may see bond prices potentially rise and equity valuations improve in late 2025 if rate cuts materialize, though near-term market volatility could persist as markets await further inflation data.


Difference
With TickerWithout TickerIMPACT line"NVDA investors benefit from accelerating AI chip demand...""Investors may see bond prices rise..."    